# Лабораторная работа №1: Первичное исследование данных

## 1. Постановка задачи

### Описание датасета
Данный датасет представляет собой исторические котировки акций компаний группы FAANG (Meta, Apple, Amazon, Netflix, Google) дополненные рассчитанными техническими индикаторами (например, скользящие средние, RSI и др.). Данные охватывают ежедневные показатели: цены открытия, закрытия, максимумы, минимумы и объемы торгов.

### Условный заказчик
Аналитический отдел инвестиционного фонда или частный трейдер, разрабатывающий стратегию автоматизированной торговли.

### Возможные задачи ИАД
1. Описательная аналитика: Сравнение волатильности и доходности различных компаний группы FAANG за определенный период.
2. Поиск аномалий: Выявление резких ценовых скачков или падений объемов, не соответствующих рыночному тренду.

## 2. Паспорт датасета

### Загрузка данных

In [ ]:
import pandas as pd
import kagglehub

# Загрузка данных
path = kagglehub.dataset_download("vishardmehta/faang-stock-market-data-with-technical-indicators")

import os
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# Размер датасета
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")

# Анализ типов данных
print("\nПаспорт столбцов:")
descriptions = [
    "Название компании (тикер)",
    "Дата торгов",
    "Цена открытия",
    "Максимальная цена за день",
    "Минимальная цена за день",
    "Цена закрытия",
    "Объем торгов",
    "Технический индикатор (тренд)"
]
if len(descriptions) < len(df.columns):
    descriptions += ['-'] * (len(df.columns) - len(descriptions))

info_df = pd.DataFrame({
    'Тип (pandas)': df.dtypes,
    'Пропуски': df.isnull().sum(),
    'Гипотеза о смысле': descriptions
})
display(info_df)

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    print("\nТип столбца 'Date' успешно приведен к datetime.")

### Структура данных

In [ ]:
# Информация о столбцах и типах
df.info()

# Статистика по числовым признакам
df.describe()

## 3. Аудит качества данных

### 3.1. Пропуски.

In [ ]:
# Процент пропусков
missing_values = df.isnull().mean() * 100
print("Процент пропусков:\n", missing_values[missing_values > 0])

### 3.2. Дубликаты.

In [ ]:
# Проверка дубликатов
duplicates = df.duplicated().sum()
print(f"\nКоличество полных дубликатов: {duplicates}")

### 3.3. Выбросы (пример для одного признака)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Анализ выбросов (Close)
target_col = 'Close' 

Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]
print(f"Найдено выбросов для {target_col}: {len(outliers)}")

plt.figure(figsize=(10, 5))
sns.boxplot(x=df[target_col])
plt.title(f'Визуализация выбросов для {target_col}')
plt.show()

## 4. Разведочный анализ (EDA)

### 4.1. Распределение числового признака

In [ ]:
# Гистограмма цены закрытия
plt.figure(figsize=(10, 6))
sns.histplot(df['Close'], kde=True, color='blue')
plt.title('Распределение цен закрытия')
plt.show()

# Корреляция цены и объема
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Volume', y='Close', hue='Ticker' if 'Ticker' in df.columns else None)
plt.title('Зависимость цены от объема торгов')
plt.show()

### 4.2. Анализ категориального признака

In [ ]:
# Анализ по компаниям
cat_col = 'Ticker'

# Настройка стиля для наглядности
sns.set_style("whitegrid")

# 1. Boxplot: Разброс цен по компаниям
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Ticker', y='Close', palette="husl", hue='Ticker')
plt.title('Разброс цен закрытия (Close) по компаниям', fontsize=14)
plt.xlabel('Компания', fontsize=12)
plt.ylabel('Цена ($)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

# 2. Barplot: Средний объем торгов
plt.figure(figsize=(12, 6))
avg_vol = df.groupby('Ticker')['Volume'].mean().sort_values(ascending=False)
sns.barplot(x=avg_vol.index, y=avg_vol.values, palette="viridis", hue=avg_vol.index)
plt.title('Средний ежедневный объем торгов', fontsize=14)
plt.xlabel('Компания', fontsize=12)
plt.ylabel('Объем (шт.)', fontsize=12)
plt.show()

## 5. Выводы

Детали в файле `report/quality_report.md`